## Lab06: Prompt Engineering

Let's consider LLAMA as our starting point. In the following, we see a typical prompt feeding and text generation with LLAMA

In [ ]:
import torch

#detect the device available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", device) # You may need to execute "pip install torch --index-url https://download.pytorch.org/whl/cu124" (or other versions in place of the 12.4) in order to use the GPU

In [ ]:

from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, GenerationConfig

# HuggingFace token

#----------------Google Colab only---------------------
# from huggingface_hub import login
#from google.colab import userdata
#login(userdata.get('HF_TOKEN'))
#----------------Google Colab only---------------------

#----------------Local Runtime only---------------------
import os
from dotenv import load_dotenv # pip install python-dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HF_TOKEN") # Create this constant in the file .env (put it in the root dir of your project)
login(token=hf_token)
#----------------Local Runtime only---------------------


# Load the model and the tokenizer (the first time it requires a lot of time to be executed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "meta-llama/Llama-3.2-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map=device, token=hf_token)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token, clean_up_tokenization_spaces=False)


# Configure generation parameters
gen_config = GenerationConfig.from_model_config(model.config)
# gen_config.temperature = 0.3              # Regulate randomness, use with do_sample = True
# gen_config.max_length = None              # Since max_new_tokens is already set
gen_config.repetition_penalty = 2.0         # Penalize repetition
gen_config.no_repeat_ngram_size = 3         # Avoid repeating bigrams
gen_config.do_sample = False                # Set to False in order to use Greedy or Beam search
gen_config.num_beams = 3                    # Use with do_sample = False
gen_config.eos_token_id = tokenizer.eos_token_id  # End generation at EOS token
gen_config.pad_token_id = tokenizer.pad_token_id
gen_config.early_stopping = True


query = "Tell me the capital of France."
messages = [
    {'role': 'user', 'content': query},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)



#alternative 1: use the tokenizer and produce the output

def make_a_query(_prompt: str, _tokenizer, _model, _device, _gen_config, max_new_tokens:int = 200):

    # Set pad_token_id if missing
    if not hasattr(_tokenizer, "pad_token_id") or _tokenizer.pad_token_id is None:
        _tokenizer.pad_token_id = _tokenizer.eos_token_id

    _gen_config.max_new_tokens = max_new_tokens  # Limit the number of new tokens generated (e.g., a single word)


    # Tokenize the input with padding and truncation
    inputs = _tokenizer(_prompt, return_tensors="pt", padding=True, truncation=True).to(_device)

    # Compute the length of the input prompt to be able to extract the model's response later
    input_ids = inputs["input_ids"]
    prompt_length = input_ids.shape[1]

    # Generate a response
    output = _model.generate(
        input_ids,
        attention_mask=inputs["attention_mask"],
        generation_config=_gen_config,
    )

    generated_tokens = output[0, prompt_length:]    # Try to print the variable "output" in order to understand its dimensionality

    # Decode the response into human-readable text
    response = _tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return response

print("\n----------------------------------------------\n\n")
print(make_a_query(_prompt=prompt, _tokenizer=tokenizer, _model=model, _gen_config=gen_config, _device=device))


#alternative 2: use the chat template and pipelines from huggingface

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    dtype=torch.float16,
    device_map="auto",
)

gen_config.max_new_tokens = 200

outputs = text_generation_pipeline(
    prompt,
    generation_config=gen_config,
) # Question: will this second generation be influenced by the previous one?

print("\n----------------------------------------------\n\n")
print(outputs[0]["generated_text"])

### Pymupdf

Reference libraries to install: pip install openai pymupdf faiss-cpu scikit-learn

PyMuPDF is a Python library that provides tools for working with PDF files (as well as other document formats like XPS, OpenXPS, CBZ, EPUB, and FB2). It's built on the MuPDF library, a lightweight, high-performance PDF and XPS rendering engine. With PyMuPDF, you can perform various tasks like reading, creating, editing, and extracting content from PDFs, images, and annotations.

In [ ]:
#-----------Google Colab only----------------------
#!pip install pymupdf # The name of the module to be installed may vary depending on the environment where you run this code (i.e., Colab versus local python environment)
# Note: Ensure "example2.pdf" is uploaded to the Colab runtime before running this.
# If the file is missing, you will get a "RuntimeError: cannot open ..."
#--------------------------------------------------

import pymupdf

#open an example pdf
doc = pymupdf.open("example2.pdf")


# Extract text from the first page
page = doc.load_page(0)
text = page.get_text("text")  # Use 'text' mode to get raw text
print(text)


### Example: Text Summarization

Let's ask LLAMA to perform a summarization of the example PDF.

In [ ]:
# Define the prompt to ask for text summarization.
text_summarization_prompt = ""      #define your prompt here
text = ""                           #load here the FULL text of the article
prompt_str =  """{PROMPT}. article: {BODY}""".format(PROMPT=text_summarization_prompt, BODY=text)

#feed the prompt to llama
#print the result of text summarization into bullets

r1 = ""

### Adding a System Prompt

Llama was trained with a system message that set the context and persona to assume when solving a task. One of the unsung advantages of open-access models is that you have full control over the system prompt in chat applications. This is essential to specify the behavior of your chat assistant –and even imbue it with some personality– but it's unreachable in models served behind APIs.


In [ ]:
#default standard system message from the Hugging Face blog to the prompt from above
system_prompt = """You are a helpful, respectful and honest assistant. \
    Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, \
    unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses \
    are socially unbiased and positive in nature. If a question does not make any sense, or is not factually \
    coherent, explain why instead of answering something not correct. If you don't know the answer to a question, \
    please don't share false information."""

# Use the same user prompt string as before
prompt_str = ""

# Put the system prompt and the user prompt in the same chat
messages = []

# Tokenize
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Query Llama again
r2 = ""

#what changes?

### Customizing the System prompt

With Llama we have full control over the system prompt. The following experiment will instruct Llama to assume the persona of a researcher tasked with writing a concise brief.

Apply the following changes the original system prompt:
- Use the researcher persona and specify the tasks to summarize articles.
- Remove safety instructions; they are unnecessary since we ask Llama to be truthful to the article.


In [ ]:
new_system_prompt = ""

# The same user prompt string as before
prompt_str = """{PROMPT}. article: {BODY}""".format(PROMPT=text_summarization_prompt, BODY=text)


r3 = ""

### Chain-of-Thought prompting

Chain-of-thought is when a prompt is being constructed using a previous prompt answer. For our use case to extract information from text, we will first ask Llama what the article is about and then use the response to ask a second question: what problem does [what the article is about] solve?



In [ ]:
# Define a prompt to ask what the article is about

prompt_str = ""

r4 = ""

# Now embed the result of the previous prompt in a new prompt to ask what that solves

prompt_str2 = ""

r5 = ""




### Generating JSONs with Llama

Llama needs precise instructions when asking it to generate JSON. In essence, here is what works for me to get valid JSON consistently:

- Explicitly state — “ All output must be in valid JSON. Don’t add explanation beyond the JSON” in the system prompt.
- Add an “explanation” variable to the JSON example. Llama enjoys explaining its answers. Give it an outlet.
- Use the JSON as part of the instruction. See the “in_less_than_ten_words” example below.
Change “write the answer” to “output the answer.”


In [ ]:


#example addition to a prompt to deal with jsons
json_prompt_addition = "Output must be in valid JSON like the following example {{\"topic\": topic, \"explanation\": [in_less_than_ten_words]}}. Output must include only JSON."

#now generate a prompt by correctly concatenating the system prompt, the json prompt instruction, and an article
prompt_str = ""

r6 = ""

#compare the difference between the prompt with the formatting instruction and a regular prompt without formatting instructions. is there any difference?




### One-to-Many Shot Learning Prompting

One-to-Many Shot Learning is a term that refers to a type of machine learning problem where the goal is to learn to recognize many different classes of objects from only one or a few examples of each class. For example, if you have only one image of a cat and one image of a dog, can you train a model to distinguish between cats and dogs in new images? This is a challenging problem because the model has to generalize well from minimal data (source)

Important points about the prompts:

- The system prompt includes the instructions to output the answer in JSON.
- The prompt consists of a one-to-many shot learning section that starts after the end of the system prompt.
- Shot examples are represented through pairs composed by user's question and assistant's response, as reported in the template below.
- The examples are given in JSON because the answers need to be JSON.
- The JSON allows defining the response with name, type, and explanation.
- The prompt question is represented by the last user's question - assistant's response pair, where the response is blank and the last <|eot_id|> is missing.

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
SYSTEM PROMPT
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Question shot 1.
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Answer shot 1.
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Question shot 2.
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Answer shot 2.
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Final prompt.
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
```

In [ ]:
#describe all the main nouns in the example.pdf article

#use the following shot examples
shot_examples = [
    {'role': 'user', 'content': """Summarize the article below. List the different main aspects in VALID JSON, following the format of the above example.

**Article:**
The rapid growth of the semiconductor industry has transformed global manufacturing, as companies specializing in advanced chip design and microdevice fabrication struggle to meet soaring technology demands."""},
    {'role': 'assistant', 'content': """[
    {"name": "semiconductor", "type": "industry", "explanation": "Companies engaged in the design and fabrication of semiconductors and semiconductor devices"},
]"""},

    {'role': 'user', 'content': """**Article:**
The NBA continues to captivate millions of fans worldwide as the premier global stage for professional basketball, driving massive television ratings and international merchandise sales each season."""},
    {'role': 'assistant', 'content': """[
    {"name": "NBA", "type": "sport league", "explanation": "NBA is the national basketball league"}
]"""},
]

#now build the prompt following the template described above
prompt_str = ""

r7 = ""

#compare the response of the prompt described above and a zero-shot prompt. Are there any differences?
